# Dates and Times

## Quick notes
### What this covers
- How SQLite stores dates (it has **no date type**): ISO-8601 TEXT, Julian day REAL, or Unix time INTEGER.
- The five date functions `date()`, `time()`, `datetime()`, `julianday()`, `strftime()` plus `unixepoch()`, and the modifiers (`'+1 month'`, `'start of month'`, `'weekday 1'`).
- Date arithmetic: differences, age, month ends, week numbers, quarters, grouping, business days, a calendar table from a recursive CTE.
- Pitfalls: invalid dates, month overflow, comparing text, time zones.
- Every demo uses **fixed dates**, never `'now'`, so every run prints the same result.
### Key points
- Store dates as TEXT in ISO format `YYYY-MM-DD` (or `YYYY-MM-DD HH:MM:SS`). It sorts and compares correctly as plain text.
- Functions accept ISO text, a Julian day number, or (with the `'unixepoch'` modifier) a Unix timestamp.
- Modifiers are applied **left to right**, so order matters.
- Subtract days with `julianday(a) - julianday(b)`. There is no `DATEDIFF`.
- Last day of a month: `date(d, 'start of month', '+1 month', '-1 day')`.
- An unreadable date gives `NULL`, not an error.
### Gotchas (the part worth revising twice)
- `date('2024-01-31', '+1 month')` is `2024-03-02`, not Feb 29: SQLite rolls the excess days over into the next month.
- `date('2024-02-30')` does **not** return `NULL`: it quietly becomes `2024-03-01`. Only truly impossible fields (month 13, day 32) give `NULL`.
- `'2024-9-5'` (no zero padding) is not a valid date, and as text it sorts after `'2024-10-01'`. Always zero-pad.
- `BETWEEN '2024-03-01' AND '2024-03-31'` misses rows on March 31 that carry a time. Use a half-open range: `>= start AND < next_start`.
- `'now'`, `'localtime'` and `'utc'` depend on the clock and the machine's time zone. They are explained here but never run.
- `strftime('%W')` week numbers are not ISO weeks, and `%V` and `%U` return `NULL` in this version.
### When it matters
- Reports by month/quarter, ageing and SLAs, subscriptions and renewals, anything that crosses a month end, a leap day or a time zone.
### One-line recap example
```sql
SELECT date('2024-01-31', 'start of month', '+1 month', '-1 day');   -- 2024-01-31, the last day of January
```

## 01. Seed data
- `orders` has an order date and a ship date (a plain TEXT date, `NULL` while unshipped). The dates deliberately include a leap day (Feb 29), month ends (Jan 31, Mar 31) and an unshipped order.
- `events` holds timestamps with a time of day, used later to show the `BETWEEN` trap.
- `people` has birth dates, including someone born on a leap day.
- "Today" in every example is a fixed date, `2024-06-15`.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE orders (
    id           INTEGER PRIMARY KEY,
    customer     TEXT NOT NULL,
    placed_on    TEXT NOT NULL,
    shipped_on   TEXT,
    amount_cents INTEGER NOT NULL
);
INSERT INTO orders VALUES
 (1, 'Acme', '2024-01-05', '2024-01-08', 12000),
 (2, 'Bolt', '2024-01-31', '2024-02-02',  4500),
 (3, 'Acme', '2024-02-14', '2024-02-14',   800),
 (4, 'Cora', '2024-02-29', NULL,         15000),
 (5, 'Bolt', '2024-03-15', '2024-03-20', 22000),
 (6, 'Cora', '2024-03-31', '2024-04-03',  3000),
 (7, 'Acme', '2024-04-10', '2024-04-12',  9900),
 (8, 'Dune', '2024-06-14', NULL,           700);
CREATE TABLE events (id INTEGER PRIMARY KEY, happened_at TEXT NOT NULL);
INSERT INTO events VALUES
 (1, '2024-03-30 23:59:00'), (2, '2024-03-31 09:00:00'),
 (3, '2024-03-31 17:45:00'), (4, '2024-04-01 00:00:00');
CREATE TABLE people (name TEXT PRIMARY KEY, born TEXT NOT NULL);
INSERT INTO people VALUES
 ('Ada', '1990-12-10'), ('Bob', '2000-02-29'), ('Cy', '1985-06-15'), ('Di', '2010-06-16');
SELECT * FROM orders ORDER BY id;

id | customer | placed_on  | shipped_on | amount_cents
---+----------+------------+------------+-------------
1  | Acme     | 2024-01-05 | 2024-01-08 | 12000       
2  | Bolt     | 2024-01-31 | 2024-02-02 | 4500        
3  | Acme     | 2024-02-14 | 2024-02-14 | 800         
4  | Cora     | 2024-02-29 | NULL       | 15000       
5  | Bolt     | 2024-03-15 | 2024-03-20 | 22000       
6  | Cora     | 2024-03-31 | 2024-04-03 | 3000        
7  | Acme     | 2024-04-10 | 2024-04-12 | 9900        
8  | Dune     | 2024-06-14 | NULL       | 700         
(8 rows)



## 02. There is no date type: three ways to store the same moment
- SQLite has only five storage classes (see 01-Basics/01-Fundamentals). Dates are just values with a convention.
- **TEXT**: ISO-8601 like `2024-03-15 08:30:00`. Human readable, sorts correctly, the recommended choice.
- **REAL**: a Julian day number, days since noon on 24 Nov 4714 BC (UTC). Good for day arithmetic.
- **INTEGER**: Unix time, seconds since 1970-01-01 00:00:00 UTC. Compact, easy to compare and add.
- The same functions convert between them. Pick **one** convention per column and stick to it.

In [3]:
%%sql
SELECT '2024-03-15 08:30:00'                                  AS iso_text,
       typeof('2024-03-15 08:30:00')                          AS t1,
       julianday('2024-03-15 08:30:00')                       AS julian_real,
       typeof(julianday('2024-03-15 08:30:00'))               AS t2,
       unixepoch('2024-03-15 08:30:00')                       AS unix_int,
       typeof(unixepoch('2024-03-15 08:30:00'))               AS t3,
       datetime(1710491400, 'unixepoch')                      AS back_from_unix,
       datetime(2460384.8541666665)                           AS back_from_julian;

iso_text            | t1   | julian_real        | t2   | unix_int   | t3      | back_from_unix      | back_from_julian   
--------------------+------+--------------------+------+------------+---------+---------------------+--------------------
2024-03-15 08:30:00 | text | 2460384.8541666665 | real | 1710491400 | integer | 2024-03-15 08:30:00 | 2024-03-15 08:30:00
(1 row)



## 03. date(), time(), datetime()
- `date(x)` gives `YYYY-MM-DD`, `time(x)` gives `HH:MM:SS`, `datetime(x)` gives `YYYY-MM-DD HH:MM:SS`. All return TEXT.
- Accepted input formats: `YYYY-MM-DD`, `YYYY-MM-DD HH:MM`, `YYYY-MM-DD HH:MM:SS`, optional fractions, the `T` separator, and a trailing `Z` or `+HH:MM` zone.
- These function calls **strip** the parts you do not ask for: `date('2024-03-15 08:30:00')` is just the date.
- Not accepted: `2024-3-5`, `15/03/2024`, `20240315`. They give `NULL` (see section 17 for parsing them).

In [4]:
%%sql
SELECT date('2024-03-15 08:30:45')           AS d,
       time('2024-03-15 08:30:45')           AS t,
       datetime('2024-03-15T08:30:45')       AS dt_with_T,
       time('2024-03-15 08:30')              AS seconds_added,
       date('2024-3-5')                      AS no_padding,
       date('15/03/2024')                    AS day_first,
       date('20240315')                      AS compact;

d          | t        | dt_with_T           | seconds_added | no_padding | day_first | compact
-----------+----------+---------------------+---------------+------------+-----------+--------
2024-03-15 | 08:30:45 | 2024-03-15 08:30:45 | 08:30:00      | NULL       | NULL      | NULL   
(1 row)



## 04. strftime(): the Swiss army knife
- `strftime(format, date, modifier...)` formats a date. Codes: `%Y` year, `%m` month, `%d` day, `%H` hour (24h), `%M` minute, `%S` second, `%j` day of year, `%w` weekday (0 = Sunday), `%W` week of year (Monday first), `%s` Unix seconds, `%f` seconds with fraction.
- It returns TEXT, even for digits: `strftime('%Y', d)` is `'2024'`, not the number. Cast it with `CAST(... AS INTEGER)` before doing maths.
- Common uses: group by month `strftime('%Y-%m', d)`, extract the year, build a label.
- The other functions are shortcuts: `date(x)` is `strftime('%Y-%m-%d', x)`.

In [5]:
%%sql
SELECT strftime('%Y', '2024-03-15 08:30:45')      AS year_text,
       typeof(strftime('%Y', '2024-03-15'))       AS year_type,
       strftime('%Y-%m', '2024-03-15')            AS year_month,
       strftime('%d/%m/%Y', '2024-03-15')         AS day_first_label,
       strftime('%H:%M', '2024-03-15 08:30:45')   AS hh_mm,
       strftime('%j', '2024-03-15')               AS day_of_year,
       strftime('%w', '2024-03-15')               AS weekday_num,
       strftime('%s', '2024-03-15')               AS unix_text,
       CAST(strftime('%Y', '2024-03-15') AS INTEGER) + 1 AS next_year;

year_text | year_type | year_month | day_first_label | hh_mm | day_of_year | weekday_num | unix_text  | next_year
----------+-----------+------------+-----------------+-------+-------------+-------------+------------+----------
2024      | text      | 2024-03    | 15/03/2024      | 08:30 | 075         | 5           | 1710460800 | 2025     
(1 row)



## 05. Modifiers: adding and subtracting time
- Extra arguments after the date are modifiers: `'+7 days'`, `'-3 months'`, `'+1 year'`, `'+90 minutes'`, `'+36 hours'`.
- Units: seconds, minutes, hours, days, months, years (the plural `s` is optional).
- `'start of month'`, `'start of year'`, `'start of day'` cut the value back to the beginning of that period.
- They return the same kind of value the function does: `date()` returns a date, `datetime()` a timestamp.

In [6]:
%%sql
SELECT date('2024-03-15', '+7 days')                  AS plus_week,
       date('2024-03-15', '-3 months')                AS minus_3_months,
       date('2024-03-15', '+1 year')                  AS plus_year,
       datetime('2024-03-15 10:00:00', '+90 minutes') AS plus_90_min,
       time('2024-03-15 10:00:00', '+36 hours')       AS plus_36_hours,
       date('2024-03-15', 'start of month')           AS month_start,
       date('2024-03-15', 'start of year')            AS year_start,
       datetime('2024-03-15 10:11:12', 'start of day') AS day_start;

plus_week  | minus_3_months | plus_year  | plus_90_min         | plus_36_hours | month_start | year_start | day_start          
-----------+----------------+------------+---------------------+---------------+-------------+------------+--------------------
2024-03-22 | 2023-12-15     | 2025-03-15 | 2024-03-15 11:30:00 | 22:00:00      | 2024-03-01  | 2024-01-01 | 2024-03-15 00:00:00
(1 row)



## 06. Modifier order matters
- Modifiers run left to right, each acting on the result of the one before.
- `'start of month', '+1 month', '-1 day'` = first of this month, then first of next month, then the day before: the **last day of this month**.
- Swap the order and you get a different answer: `'+1 month', 'start of month'` is just the first of next month.
- Think of it as a pipeline, and read it in order.

In [7]:
%%sql
SELECT date('2024-03-15', 'start of month', '+1 month', '-1 day') AS last_day_this_month,
       date('2024-03-15', '+1 month', 'start of month')           AS first_of_next_month,
       date('2024-03-15', 'start of month', '+1 month')           AS same_as_above,
       date('2024-03-15', 'start of year', '+1 year', '-1 day')   AS last_day_of_year,
       date('2024-03-15', '-1 day', 'start of month')             AS first_of_month_of_yesterday;

last_day_this_month | first_of_next_month | same_as_above | last_day_of_year | first_of_month_of_yesterday
--------------------+---------------------+---------------+------------------+----------------------------
2024-03-31          | 2024-04-01          | 2024-04-01    | 2024-12-31       | 2024-03-01                 
(1 row)



## 07. Weekdays: 'weekday N' and the name of the day
- `strftime('%w', d)` gives 0 for Sunday up to 6 for Saturday.
- The modifier `'weekday N'` moves **forward** to the next date that falls on weekday N (staying put if it already does). Monday = 1, Sunday = 0.
- Next Monday after a date: `date(d, '+1 day', 'weekday 1')` always moves strictly ahead. Without the `+1 day`, a Monday stays on itself.
- Start of the week (Monday): go back 6 days, then forward to Monday: `date(d, '-6 days', 'weekday 1')`.
- There is no function that returns a day name. Take it from a string with `substr`: `substr('SunMonTueWedThuFriSat', w * 3 + 1, 3)`.

In [8]:
%%sql
SELECT d,
       substr('SunMonTueWedThuFriSat', CAST(strftime('%w', d) AS INTEGER) * 3 + 1, 3) AS day_name,
       date(d, 'weekday 1')                  AS next_or_same_monday,
       date(d, '+1 day', 'weekday 1')        AS next_monday_strict,
       date(d, '-6 days', 'weekday 1')       AS week_start_monday,
       date(d, 'weekday 0')                  AS next_or_same_sunday
FROM (SELECT '2024-05-13' AS d UNION ALL SELECT '2024-05-15' UNION ALL SELECT '2024-05-19')
ORDER BY d;

d          | day_name | next_or_same_monday | next_monday_strict | week_start_monday | next_or_same_sunday
-----------+----------+---------------------+--------------------+-------------------+--------------------
2024-05-13 | Mon      | 2024-05-13          | 2024-05-20         | 2024-05-13        | 2024-05-19         
2024-05-15 | Wed      | 2024-05-20          | 2024-05-20         | 2024-05-13        | 2024-05-19         
2024-05-19 | Sun      | 2024-05-20          | 2024-05-20         | 2024-05-13        | 2024-05-19         
(3 rows)



## 08. Month-end overflow: the biggest date trap
- Adding a month to the 31st lands on a day that does not exist, so SQLite carries the extra days into the next month.
- `2024-01-31 + 1 month` is "Feb 31" = `2024-03-02` (2024 is a leap year, so February has 29 days: Feb 31 overflows by 2 days).
- Same with `2024-03-31 - 1 month` = "Feb 31" = `2024-03-02`, and `2024-02-29 + 1 year` = `2025-03-01`.
- If you want "same day next month, but clamp to the month end", compute both candidates and take the smaller: `min(date(d, '+1 month'), last day of next month)`.
- Subscription billing and loan schedules bite here. Decide the rule, then write it once (in a view or the application).

In [9]:
%%sql
SELECT date('2024-01-31', '+1 month')    AS jan31_plus_1m,
       date('2024-03-31', '-1 month')    AS mar31_minus_1m,
       date('2024-02-29', '+1 year')     AS leap_plus_1y,
       date('2024-01-30', '+1 month')    AS jan30_plus_1m;
SELECT d,
       date(d, '+1 month')                                                 AS naive,
       min(date(d, '+1 month'), date(d, 'start of month', '+2 months', '-1 day')) AS clamped
FROM (SELECT '2024-01-31' AS d UNION ALL SELECT '2024-03-31' UNION ALL SELECT '2024-05-15')
ORDER BY d;

jan31_plus_1m | mar31_minus_1m | leap_plus_1y | jan30_plus_1m
--------------+----------------+--------------+--------------
2024-03-02    | 2024-03-02     | 2025-03-01   | 2024-03-01   
(1 row)

d          | naive      | clamped   
-----------+------------+-----------
2024-01-31 | 2024-03-02 | 2024-02-29
2024-03-31 | 2024-05-01 | 2024-04-30
2024-05-15 | 2024-06-15 | 2024-06-15
(3 rows)



## 09. Last day of month and days in month
- Last day: `date(d, 'start of month', '+1 month', '-1 day')`.
- Days in the month: the day number of that last day, `CAST(strftime('%d', last_day) AS INTEGER)`.
- Leap years come out right for free: February 2024 has 29 days, February 2023 has 28.
- Orders placed on the last day of their month: compare the order date with its own last day.

In [10]:
%%sql
SELECT d,
       date(d, 'start of month', '+1 month', '-1 day')                                AS last_day,
       CAST(strftime('%d', date(d, 'start of month', '+1 month', '-1 day')) AS INTEGER) AS days_in_month
FROM (SELECT '2024-02-10' AS d UNION ALL SELECT '2023-02-10' UNION ALL SELECT '2024-04-30' UNION ALL SELECT '2024-12-01')
ORDER BY d;
SELECT id, placed_on
FROM orders
WHERE placed_on = date(placed_on, 'start of month', '+1 month', '-1 day')
ORDER BY id;

d          | last_day   | days_in_month
-----------+------------+--------------
2023-02-10 | 2023-02-28 | 28           
2024-02-10 | 2024-02-29 | 29           
2024-04-30 | 2024-04-30 | 30           
2024-12-01 | 2024-12-31 | 31           
(4 rows)

id | placed_on 
---+-----------
2  | 2024-01-31
4  | 2024-02-29
6  | 2024-03-31
(3 rows)



## 10. Differences: days, hours, minutes, seconds
- There is no `DATEDIFF`. Subtract Julian days: `julianday(b) - julianday(a)` is a real number of days (with a fraction for the time of day).
- For whole days, strip the time first (use `date()`) or cast the result, since a time of day adds a fraction.
- Hours / minutes / seconds: multiply by 24 / 1440 / 86400, or subtract `unixepoch()` values to get exact integer seconds.
- `timediff(a, b)` (SQLite 3.43+) returns a readable `+YYYY-MM-DD HH:MM:SS.SSS` text value, but months have different lengths, so use it for display, not maths.
- Days to ship per order, `NULL` while unshipped (`NULL` minus anything is `NULL`):

In [11]:
%%sql
SELECT id, placed_on, shipped_on,
       CAST(julianday(shipped_on) - julianday(placed_on) AS INTEGER) AS days_to_ship
FROM orders
ORDER BY id;
SELECT julianday('2024-03-15 18:00:00') - julianday('2024-03-15 06:00:00')            AS days_fraction,
       (julianday('2024-03-15 18:00:00') - julianday('2024-03-15 06:00:00')) * 24     AS hours,
       unixepoch('2024-03-15 18:00:00') - unixepoch('2024-03-15 06:00:00')            AS seconds,
       julianday('2024-03-01') - julianday('2024-02-01')                              AS feb_length,
       timediff('2024-03-01', '2024-01-15')                                            AS readable;

id | placed_on  | shipped_on | days_to_ship
---+------------+------------+-------------
1  | 2024-01-05 | 2024-01-08 | 3           
2  | 2024-01-31 | 2024-02-02 | 2           
3  | 2024-02-14 | 2024-02-14 | 0           
4  | 2024-02-29 | NULL       | NULL        
5  | 2024-03-15 | 2024-03-20 | 5           
6  | 2024-03-31 | 2024-04-03 | 3           
7  | 2024-04-10 | 2024-04-12 | 2           
8  | 2024-06-14 | NULL       | NULL        
(8 rows)

days_fraction | hours | seconds | feb_length | readable                
--------------+-------+---------+------------+-------------------------
0.5           | 12.0  | 43200   | 29.0       | +0000-01-15 00:00:00.000
(1 row)



## 11. Age and "full years between"
- Subtracting the years is wrong before the birthday has happened this year.
- Correct: year difference, minus 1 if the month-day of the as-of date is earlier than the month-day of the birth date. Comparing the `'MM-DD'` strings does this neatly.
- We use a fixed as-of date, `2024-06-15`. Cy's birthday is that very day, so Cy has already turned 39; Di's is tomorrow, so Di is still 13.
- Born on Feb 29 (Bob): the `'02-29'` string is less than `'06-15'`, so the birthday counts as passed. On non-leap years the same rule treats Mar 1 as the birthday.

In [12]:
%%sql
SELECT name, born,
       CAST(strftime('%Y', '2024-06-15') AS INTEGER) - CAST(strftime('%Y', born) AS INTEGER)
         - (strftime('%m-%d', '2024-06-15') < strftime('%m-%d', born)) AS age,
       CAST(strftime('%Y', '2024-06-15') AS INTEGER) - CAST(strftime('%Y', born) AS INTEGER) AS naive_age
FROM people
ORDER BY name;

name | born       | age | naive_age
-----+------------+-----+----------
Ada  | 1990-12-10 | 33  | 34       
Bob  | 2000-02-29 | 24  | 24       
Cy   | 1985-06-15 | 39  | 39       
Di   | 2010-06-16 | 13  | 14       
(4 rows)



## 12. Week numbers
- `%W` = week of the year, Monday as the first day, days before the first Monday are week `00` (see 2021-01-03 below). `%U` (Sunday first) is not available in this version either (it returns `NULL`), so avoid it.
- These are **not** ISO weeks. ISO weeks (used in Europe and in many reports) are weeks that belong to the year containing their Thursday, so Monday 2024-12-30 is already ISO week 1 of 2025. `%V` is not supported in this version.
- ISO week by the Thursday trick: find the Thursday of the same Monday-Sunday week, `date(d, '-3 days', 'weekday 4')`; its year is the ISO year and `(day of year - 1) / 7 + 1` is the ISO week.
- Rule: pick one definition, label it in the report, and keep it everywhere.

In [13]:
%%sql
SELECT d,
       strftime('%W', d) AS w_monday_first,
       strftime('%Y', date(d, '-3 days', 'weekday 4'))                                  AS iso_year,
       (CAST(strftime('%j', date(d, '-3 days', 'weekday 4')) AS INTEGER) - 1) / 7 + 1   AS iso_week
FROM (SELECT '2024-01-01' AS d UNION ALL SELECT '2024-01-07' UNION ALL SELECT '2024-06-15'
      UNION ALL SELECT '2024-12-29' UNION ALL SELECT '2024-12-30' UNION ALL SELECT '2021-01-03')
ORDER BY d;

d          | w_monday_first | iso_year | iso_week
-----------+----------------+----------+---------
2021-01-03 | 00             | 2020     | 53      
2024-01-01 | 01             | 2024     | 1       
2024-01-07 | 01             | 2024     | 1       
2024-06-15 | 24             | 2024     | 24      
2024-12-29 | 52             | 2024     | 52      
2024-12-30 | 53             | 2025     | 1       
(6 rows)



## 13. Quarter and fiscal year
- SQLite has no quarter code. Two ways: a `CASE` on the month, or integer arithmetic `(month + 2) / 3` (integer division, see 04-Math-And-Numbers).
- `CASE` is easiest to read and lets you shift to a fiscal calendar (for example, a year starting in April).
- Fiscal quarter with an April start: months 4-6 are Q1, 7-9 Q2, 10-12 Q3, 1-3 Q4. The fiscal year is named by the year it ends in.

In [14]:
%%sql
SELECT id, placed_on,
       CASE WHEN m BETWEEN 1 AND 3 THEN 'Q1'
            WHEN m BETWEEN 4 AND 6 THEN 'Q2'
            WHEN m BETWEEN 7 AND 9 THEN 'Q3'
            ELSE 'Q4' END                          AS quarter_case,
       'Q' || ((m + 2) / 3)                        AS quarter_math,
       CASE WHEN m >= 4 THEN 'FY' || (y + 1) ELSE 'FY' || y END AS fiscal_year_apr,
       'FQ' || ((m + 8) % 12 / 3 + 1)              AS fiscal_quarter_apr
FROM (SELECT id, placed_on,
             CAST(strftime('%m', placed_on) AS INTEGER) AS m,
             CAST(strftime('%Y', placed_on) AS INTEGER) AS y
      FROM orders)
ORDER BY id;

id | placed_on  | quarter_case | quarter_math | fiscal_year_apr | fiscal_quarter_apr
---+------------+--------------+--------------+-----------------+-------------------
1  | 2024-01-05 | Q1           | Q1           | FY2024          | FQ4               
2  | 2024-01-31 | Q1           | Q1           | FY2024          | FQ4               
3  | 2024-02-14 | Q1           | Q1           | FY2024          | FQ4               
4  | 2024-02-29 | Q1           | Q1           | FY2024          | FQ4               
5  | 2024-03-15 | Q1           | Q1           | FY2024          | FQ4               
6  | 2024-03-31 | Q1           | Q1           | FY2024          | FQ4               
7  | 2024-04-10 | Q2           | Q2           | FY2025          | FQ1               
8  | 2024-06-14 | Q2           | Q2           | FY2025          | FQ1               
(8 rows)



## 14. Grouping by month, week and quarter
- Group on a derived text key: `strftime('%Y-%m', placed_on)`. Include the year or you will mix January 2023 and January 2024.
- Months with no orders **do not appear** in a plain `GROUP BY` (here: May). To show them, left-join against a calendar (section 20).
- The same pattern works for quarters, weeks (`strftime('%Y-%W', d)`), or "first of the month" (`date(d, 'start of month')`), which stays a real date and sorts nicely.
- Totals in cents, formatted at the end (see 04-Math-And-Numbers).

In [15]:
%%sql
SELECT strftime('%Y-%m', placed_on)  AS month,
       count(*)                      AS orders,
       sum(amount_cents)             AS cents
FROM orders
GROUP BY strftime('%Y-%m', placed_on)
ORDER BY month;
SELECT date(placed_on, 'start of month') AS month_start,
       printf('%.2f', sum(amount_cents) / 100.0) AS dollars
FROM orders
GROUP BY date(placed_on, 'start of month')
ORDER BY month_start;

month   | orders | cents
--------+--------+------
2024-01 | 2      | 16500
2024-02 | 2      | 15800
2024-03 | 2      | 25000
2024-04 | 1      | 9900 
2024-06 | 1      | 700  
(5 rows)

month_start | dollars
------------+--------
2024-01-01  | 165.00 
2024-02-01  | 158.00 
2024-03-01  | 250.00 
2024-04-01  | 99.00  
2024-06-01  | 7.00   
(5 rows)



## 15. Comparing dates as text, and the BETWEEN trap
- ISO text compares correctly because year, month and day are most significant first and zero padded. `'2024-03-15' < '2024-04-01'` is true as plain string comparison.
- A non-padded date breaks it: `'2024-9-30'` sorts **after** `'2024-10-05'` because `'9'` > `'1'`.
- `date <= '2024-03-31'` works on date columns, but on timestamps `'2024-03-31 09:00:00'` is **greater** than `'2024-03-31'` (longer string, same prefix), so those rows fall out of the range.
- Safe rule: **half-open range** `>= start AND < next_start`. It never needs "the last second of the day", and it is index friendly.

In [16]:
%%sql
SELECT '2024-03-15' < '2024-04-01' AS iso_ok,
       '2024-9-30' > '2024-10-05'  AS unpadded_breaks,
       '2024-03-31 09:00:00' > '2024-03-31' AS timestamp_gt_date;
SELECT id, happened_at FROM events
WHERE happened_at BETWEEN '2024-03-01' AND '2024-03-31'
ORDER BY id;
SELECT id, happened_at FROM events
WHERE happened_at >= '2024-03-01' AND happened_at < '2024-04-01'
ORDER BY id;

iso_ok | unpadded_breaks | timestamp_gt_date
-------+-----------------+------------------
1      | 1               | 1                
(1 row)

id | happened_at        
---+--------------------
1  | 2024-03-30 23:59:00
(1 row)

id | happened_at        
---+--------------------
1  | 2024-03-30 23:59:00
2  | 2024-03-31 09:00:00
3  | 2024-03-31 17:45:00
(3 rows)



## 16. Invalid dates and what comes back
- A string that cannot be read as a date gives `NULL` (no error). That also means `date(col) IS NULL` finds bad data.
- Beware the quiet rollover: day numbers up to 31 are accepted for every month, so `'2024-02-30'` becomes `2024-03-01` and `'2023-02-29'` becomes `2023-03-01`.
- Truly out-of-range fields give `NULL`: month 13, day 32, hour 25.
- **Validate** with a round trip: `date(x) IS x` is `1` only for a real, normalised date and `0` for everything else. Use `IS`, not `=`: for garbage `date(x)` is `NULL`, `NULL = x` is `NULL`, and a `CHECK` lets `NULL` through. Add it as a `CHECK` constraint to stop bad dates at the door (see 05-DDL-And-Integrity).

In [17]:
%%sql
SELECT date('2024-02-30')   AS feb30_rolls_over,
       date('2023-02-29')   AS non_leap_feb29,
       date('2024-13-01')   AS month13,
       date('2024-01-32')   AS day32,
       date('2024-01-15 25:00') AS hour25,
       date('not a date')   AS garbage;
SELECT x, date(x) AS parsed, date(x) IS x AS is_clean_date
FROM (SELECT '2024-02-29' AS x UNION ALL SELECT '2024-02-30' UNION ALL SELECT '2024-2-3' UNION ALL SELECT 'oops')
ORDER BY x;

feb30_rolls_over | non_leap_feb29 | month13 | day32 | hour25 | garbage
-----------------+----------------+---------+-------+--------+--------
2024-03-01       | 2023-03-01     | NULL    | NULL  | NULL   | NULL   
(1 row)

x          | parsed     | is_clean_date
-----------+------------+--------------
2024-02-29 | 2024-02-29 | 1            
2024-02-30 | 2024-03-01 | 0            
2024-2-3   | NULL       | 0            
oops       | NULL       | 0            
(4 rows)



In [18]:
%%sql expect_error
CREATE TABLE strict_dates (d TEXT CHECK (date(d) IS d));
INSERT INTO strict_dates VALUES ('2024-02-29');
INSERT INTO strict_dates VALUES ('2024-02-30');

ERROR: CHECK constraint failed: date(d) IS d



## 17. Parsing non-ISO dates
- Formats like `15/03/2024` or `Mar 5, 2024` are not understood. Rebuild an ISO string with `substr` (and `instr` if the parts have variable length).
- For `DD/MM/YYYY`: `substr(x, 7, 4) || '-' || substr(x, 4, 2) || '-' || substr(x, 1, 2)`.
- Month names need a lookup: find the three letters in `'JanFebMar...'` with `instr` and turn the position into a month number: `(instr(...) + 2) / 3`.
- Convert once while loading, store ISO, and validate the result with `date(x) IS x`. Note `31/02/2024` becomes `2024-02-31`, which `date()` quietly rolls over to `2024-03-02`; only `date(iso) IS iso` would catch it. `99/99/2024` gives `NULL`.

In [19]:
%%sql
SELECT raw,
       substr(raw, 7, 4) || '-' || substr(raw, 4, 2) || '-' || substr(raw, 1, 2)                  AS iso,
       date(substr(raw, 7, 4) || '-' || substr(raw, 4, 2) || '-' || substr(raw, 1, 2))           AS validated
FROM (SELECT '15/03/2024' AS raw UNION ALL SELECT '05/11/2023' UNION ALL SELECT '31/02/2024' UNION ALL SELECT '99/99/2024')
ORDER BY raw;
SELECT raw,
       substr(raw, -4) || '-' ||
       printf('%02d', (instr('JanFebMarAprMayJunJulAugSepOctNovDec', substr(raw, 1, 3)) + 2) / 3) || '-' ||
       printf('%02d', CAST(substr(raw, 5, instr(raw, ',') - 5) AS INTEGER)) AS iso
FROM (SELECT 'Mar 5, 2024' AS raw UNION ALL SELECT 'Dec 25, 2023' UNION ALL SELECT 'Jun 14, 2024')
ORDER BY iso;

raw        | iso        | validated 
-----------+------------+-----------
05/11/2023 | 2023-11-05 | 2023-11-05
15/03/2024 | 2024-03-15 | 2024-03-15
31/02/2024 | 2024-02-31 | 2024-03-02
99/99/2024 | 2024-99-99 | NULL      
(4 rows)

raw          | iso       
-------------+-----------
Dec 25, 2023 | 2023-12-25
Mar 5, 2024  | 2024-03-05
Jun 14, 2024 | 2024-06-14
(3 rows)



## 18. Time zones: 'utc', 'localtime', and what to do instead
- SQLite stores **no** time zone with a date. Plain `2024-03-15 08:30:00` is just digits; its meaning is whatever you decided.
- Modifiers `'utc'` and `'localtime'` convert using the **machine's** time zone. They give different answers on different computers, so they are not shown running here.
- `'now'` reads the clock and is UTC. `datetime('now')` and `datetime('now', 'localtime')` differ by your offset. Any query using `'now'` returns something different on every run, so tests should pass the date in as a parameter.
- An explicit offset in the text *is* understood and deterministic: `2024-06-01 12:00:00+05:30` is converted to UTC.
- Advice: store UTC (text with a `Z`, or Unix integers), convert to local time in the application at display time, and keep the user's zone name in another column. Since UTC has no daylight saving, nothing is ever ambiguous or missing on the clock-change nights.

In [20]:
%%sql
SELECT datetime('2024-06-01 12:00:00+05:30')   AS ist_to_utc,
       datetime('2024-06-01T12:00:00Z')        AS z_means_utc,
       datetime('2024-06-01 12:00:00-04:00')   AS new_york_summer_to_utc,
       datetime('2024-06-01 12:00:00')         AS no_zone_taken_as_is;

ist_to_utc          | z_means_utc         | new_york_summer_to_utc | no_zone_taken_as_is
--------------------+---------------------+------------------------+--------------------
2024-06-01 06:30:00 | 2024-06-01 12:00:00 | 2024-06-01 16:00:00    | 2024-06-01 12:00:00
(1 row)



## 19. Storage advice and a safe column definition
- Preferred: TEXT `YYYY-MM-DD` for calendar dates (birthdays, due dates, order dates) and TEXT `YYYY-MM-DD HH:MM:SS` in **UTC** for moments in time.
- Unix INTEGER seconds are good for logs and event streams, and arithmetic is plain `+`. Julian REAL is rarely the best choice.
- Do not mix formats in one column: `'2024-03-15'`, `'03/15/24'` and `1710000000` together will sort and compare wrongly with no error.
- Enforce it: a `CHECK (date(col) IS col)` (section 16; `IS` so that unreadable text fails instead of slipping through as `NULL`), plus `NOT NULL` where required.
- Keep dates and times separate if you only need dates. A column of timestamps makes "all orders on March 31" a range query.

In [21]:
%%sql
CREATE TABLE appointments (
    id        INTEGER PRIMARY KEY,
    on_date   TEXT NOT NULL CHECK (date(on_date) IS on_date),
    starts_at TEXT CHECK (datetime(starts_at) IS starts_at)
);
INSERT INTO appointments VALUES (1, '2024-03-15', '2024-03-15 08:30:00'), (2, '2024-03-16', NULL);
SELECT * FROM appointments ORDER BY id;

id | on_date    | starts_at          
---+------------+--------------------
1  | 2024-03-15 | 2024-03-15 08:30:00
2  | 2024-03-16 | NULL               
(2 rows)



In [22]:
%%sql expect_error
INSERT INTO appointments VALUES (3, '15/03/2024', NULL);

ERROR: CHECK constraint failed: date(on_date) IS on_date



In [23]:
%%sql expect_error
INSERT INTO appointments VALUES (4, '2024-03-17', '2024-03-17T08:30:00');

ERROR: CHECK constraint failed: datetime(starts_at) IS starts_at



## 20. A calendar table with a recursive CTE
- A calendar is one row per day. Build it with a recursive CTE that adds one day at a time (recursive CTE basics are in 02-Querying).
- It lets you do what `GROUP BY` alone cannot: show **days or months with zero activity**, label weekends, number the weeks, and attach holidays.
- Always give the recursion a stop condition (`WHERE d < end_date`).
- Here: every day from 2024-01-29 to 2024-02-04, with a weekday name and a weekend flag, left-joined to orders so days without orders still show.

In [24]:
%%sql
WITH RECURSIVE cal(d) AS (
    SELECT '2024-01-29' UNION ALL SELECT date(d, '+1 day') FROM cal WHERE d < '2024-02-04'
)
SELECT cal.d,
       substr('SunMonTueWedThuFriSat', CAST(strftime('%w', cal.d) AS INTEGER) * 3 + 1, 3) AS day_name,
       strftime('%w', cal.d) IN ('0', '6')                                             AS is_weekend,
       count(o.id)                                                                     AS orders_placed,
       coalesce(sum(o.amount_cents), 0)                                                AS cents
FROM cal LEFT JOIN orders o ON o.placed_on = cal.d
GROUP BY cal.d
ORDER BY cal.d;
SELECT m.month_start, count(o.id) AS orders
FROM (WITH RECURSIVE mm(month_start) AS (
          SELECT '2024-01-01' UNION ALL SELECT date(month_start, '+1 month') FROM mm WHERE month_start < '2024-06-01')
      SELECT month_start FROM mm) m
LEFT JOIN orders o ON date(o.placed_on, 'start of month') = m.month_start
GROUP BY m.month_start
ORDER BY m.month_start;

d          | day_name | is_weekend | orders_placed | cents
-----------+----------+------------+---------------+------
2024-01-29 | Mon      | 0          | 0             | 0    
2024-01-30 | Tue      | 0          | 0             | 0    
2024-01-31 | Wed      | 0          | 1             | 4500 
2024-02-01 | Thu      | 0          | 0             | 0    
2024-02-02 | Fri      | 0          | 0             | 0    
2024-02-03 | Sat      | 1          | 0             | 0    
2024-02-04 | Sun      | 1          | 0             | 0    
(7 rows)

month_start | orders
------------+-------
2024-01-01  | 2     
2024-02-01  | 2     
2024-03-01  | 2     
2024-04-01  | 1     
2024-05-01  | 0     
2024-06-01  | 1     
(6 rows)



## 21. Business days
- Business days = calendar days that are not Saturday, Sunday or a holiday.
- Build the calendar, filter on `strftime('%w', d) NOT IN ('0', '6')`, and anti-join a `holidays` table.
- Counting business days between two dates is then a simple `count(*)`. The start is **exclusive** and the end **inclusive** below, like "days after the order".
- "Ship by 3 business days after the order": take the third row of that filtered calendar, `ORDER BY d LIMIT 1 OFFSET 2`.
- Real holiday lists change every year and by country; keep them as data, not as code.

In [25]:
%%sql
CREATE TABLE holidays (day TEXT PRIMARY KEY CHECK (date(day) IS day), name TEXT NOT NULL);
INSERT INTO holidays VALUES ('2024-03-29', 'Good Friday'), ('2024-04-01', 'Easter Monday');
CREATE VIEW workdays AS
WITH RECURSIVE cal(d) AS (
    SELECT '2024-03-01' UNION ALL SELECT date(d, '+1 day') FROM cal WHERE d < '2024-04-30'
)
SELECT d FROM cal
WHERE strftime('%w', d) NOT IN ('0', '6')
  AND d NOT IN (SELECT day FROM holidays);
SELECT count(*) AS business_days_after_mar15_up_to_apr12
FROM workdays WHERE d > '2024-03-15' AND d <= '2024-04-12';
SELECT d AS third_business_day_after_mar27
FROM workdays WHERE d > '2024-03-27' ORDER BY d LIMIT 1 OFFSET 2;

business_days_after_mar15_up_to_apr12
-------------------------------------
18                                   
(1 row)

third_business_day_after_mar27
------------------------------
2024-04-03                    
(1 row)



## 22. Putting it together: a monthly order report
- For each order: month key, quarter, days to ship (or "not shipped"), and whether it shipped within 3 days. Good habits: no hand-typed day counts and no `'now'`.
- Shipped late = more than 3 days between order and shipment. Unshipped orders are shown as `open`, not as on time.
- All dates are literal, so the report prints the same today and next year.

In [26]:
%%sql
SELECT id, customer, placed_on,
       strftime('%Y-%m', placed_on)                                            AS month,
       'Q' || ((CAST(strftime('%m', placed_on) AS INTEGER) + 2) / 3)           AS quarter,
       CASE WHEN shipped_on IS NULL THEN 'open'
            WHEN julianday(shipped_on) - julianday(placed_on) <= 3 THEN 'on time'
            ELSE 'late' END                                                    AS status,
       CAST(julianday(shipped_on) - julianday(placed_on) AS INTEGER)           AS days_to_ship
FROM orders
ORDER BY id;

id | customer | placed_on  | month   | quarter | status  | days_to_ship
---+----------+------------+---------+---------+---------+-------------
1  | Acme     | 2024-01-05 | 2024-01 | Q1      | on time | 3           
2  | Bolt     | 2024-01-31 | 2024-01 | Q1      | on time | 2           
3  | Acme     | 2024-02-14 | 2024-02 | Q1      | on time | 0           
4  | Cora     | 2024-02-29 | 2024-02 | Q1      | open    | NULL        
5  | Bolt     | 2024-03-15 | 2024-03 | Q1      | late    | 5           
6  | Cora     | 2024-03-31 | 2024-03 | Q1      | on time | 3           
7  | Acme     | 2024-04-10 | 2024-04 | Q2      | on time | 2           
8  | Dune     | 2024-06-14 | 2024-06 | Q2      | open    | NULL        
(8 rows)

